# Krones Final Evaluation Notebook

This notebook is the final evaluation target. It is intentionally standalone:
it loads the attached ONNX model, reads the competition test data, runs ROI-based
inference, and writes `submission.csv`.

It should not depend on the local project scripts, `.venv`, training outputs, or
internet downloads.


## Expected Kaggle Inputs

Attach these inputs to the Kaggle evaluation notebook:

- competition data containing `sample_submission.csv`, `test_images/`, and
  `test_annotations_roi_only.json`
- `bottletypes.csv` from the competition data, used as the second ONNX input
- model dataset containing `final_effnetv2_s.onnx`
- optional `model_metadata.json` next to the ONNX file

The metadata file stores preprocessing and threshold values. The notebook has
safe defaults, but the final trained model should provide final calibrated
values.


In [1]:
from pathlib import Path
import json
import os
import time

import cv2
import numpy as np
import pandas as pd

try:
    import onnxruntime as ort
except ImportError as exc:
    raise ImportError(
        "onnxruntime is required for the final ONNX evaluation notebook. "
        "Use the competition starter environment or attach an offline wheel if needed."
    ) from exc


IMG_SIZE = 256
BATCH_SIZE = 64
DEFAULT_THRESHOLD = 0.5
DEFAULT_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
DEFAULT_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
MODEL_FILENAMES = ["final_effnetv2_s.onnx", "model.onnx"]
N_BOTTLE_TYPES = 3

print("onnxruntime:", ort.__version__)
print("available providers:", ort.get_available_providers())


onnxruntime: 1.26.0
available providers: ['CoreMLExecutionProvider', 'AzureExecutionProvider', 'CPUExecutionProvider']


In [2]:
def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/1st-krones-vision-ai-challenge"),
        Path("/kaggle/input"),
        Path("data"),
        Path("../data"),
        Path.cwd().parent / "data",
        Path.cwd(),
    ]
    for base in candidates:
        if not base.exists():
            continue
        for current, dirs, files in os.walk(base):
            current_path = Path(current)
            if "sample_submission.csv" in files and (
                (current_path / "test_images").exists()
                or "test_annotations_roi_only.json" in files
            ):
                return current_path
    raise FileNotFoundError(
        "Could not find competition data root with sample_submission.csv."
    )


def find_model_path() -> Path:
    candidates = [
        Path("/kaggle/input"),
        Path("model"),
        Path("final_submission/model"),
        Path("../final_submission/model"),
        Path("outputs/final_effnetv2_s"),
        Path("../outputs/final_effnetv2_s"),
        Path.cwd(),
    ]
    for base in candidates:
        if not base.exists():
            continue
        for name in MODEL_FILENAMES:
            direct = base / name
            if direct.exists():
                return direct
        for current, dirs, files in os.walk(base):
            for name in MODEL_FILENAMES:
                if name in files:
                    return Path(current) / name
    raise FileNotFoundError(
        "Could not find final_effnetv2_s.onnx. Attach the trained model as a Kaggle input."
    )


def find_bottletypes_path(data_root: Path) -> Path:
    direct = data_root / "bottletypes.csv"
    if direct.exists():
        return direct
    for base in [data_root, Path("/kaggle/input")]:
        if not base.exists():
            continue
        for current, dirs, files in os.walk(base):
            if "bottletypes.csv" in files:
                return Path(current) / "bottletypes.csv"
    return direct


DATA_ROOT = find_data_root()
MODEL_PATH = find_model_path()
MODEL_DIR = MODEL_PATH.parent

SAMPLE_PATH = DATA_ROOT / "sample_submission.csv"
TEST_IMG_DIR = DATA_ROOT / "test_images"
ROI_PATH = DATA_ROOT / "test_annotations_roi_only.json"
BOTTLE_PATH = find_bottletypes_path(DATA_ROOT)

print("DATA_ROOT:", DATA_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("TEST_IMG_DIR:", TEST_IMG_DIR)
print("ROI_PATH:", ROI_PATH if ROI_PATH.exists() else "not found; full-image fallback")
print("BOTTLE_PATH:", BOTTLE_PATH if BOTTLE_PATH.exists() else "not found; unknown-type fallback")


DATA_ROOT: ../data
MODEL_PATH: model/final_effnetv2_s.onnx
TEST_IMG_DIR: ../data/test_images
ROI_PATH: ../data/test_annotations_roi_only.json
BOTTLE_PATH: ../data/bottletypes.csv


In [3]:
def load_metadata(model_dir: Path) -> dict:
    path = model_dir / "model_metadata.json"
    if path.exists():
        with path.open() as handle:
            return json.load(handle)
    return {}


metadata = load_metadata(MODEL_DIR)
IMG_SIZE = int(metadata.get("img_size", IMG_SIZE))
BATCH_SIZE = int(metadata.get("batch_size", BATCH_SIZE))
THRESHOLD = float(metadata.get("threshold", DEFAULT_THRESHOLD))
OUTPUT_ACTIVATION = metadata.get("output_activation", "sigmoid")
NORM_MEAN = np.array(metadata.get("mean", DEFAULT_MEAN.tolist()), dtype=np.float32)
NORM_STD = np.array(metadata.get("std", DEFAULT_STD.tolist()), dtype=np.float32)
INPUT_NAMES = metadata.get("input_names", ["image"])

print("metadata:", metadata or "defaults")
print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("THRESHOLD:", THRESHOLD)
print("OUTPUT_ACTIVATION:", OUTPUT_ACTIVATION)
print("INPUT_NAMES:", INPUT_NAMES)


metadata: {'model_name': 'tf_efficientnetv2_s.in21k_ft_in1k', 'model_file': 'final_effnetv2_s.onnx', 'img_size': 256, 'batch_size': 32, 'threshold': 0.6150000000000004, 'output_activation': 'sigmoid', 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225], 'input_names': ['image', 'btype_id'], 'onnx_opset': 18, 'uses_roi_crop': True, 'uses_bottle_type_metadata': True, 'training_label_source': 'COCO-derived target from train_annotations.json via outputs/preprocessing/final_train.csv', 'teacher_distillation': 'ConvNeXt Small + MaxViT Tiny soft probabilities when available', 'robust_noisy_label_weighting': 'hard-label loss is downweighted when teacher ensemble strongly disagrees with COCO target', 'feature_distillation_dirs': ['outputs/feature_distillation/lb_convnext_small', 'outputs/feature_distillation/lb_maxvit'], 'feature_weight': 0.75, 'aux27_weight': 0.2, 'rkd_weight': 0.05, 'validation_f1': 0.9642248722316865, 'validation_epoch': 5}
IMG_SIZE: 256
BATCH_SIZE: 32
THRESHOLD: 0.6

In [4]:
def load_roi_map(path: Path) -> dict[str, list[float]]:
    if not path.exists():
        return {}
    with path.open() as handle:
        coco = json.load(handle)
    id_to_name = {
        int(image["id"]): image["file_name"]
        for image in coco.get("images", [])
    }
    roi_map = {}
    for ann in coco.get("annotations", []):
        name = id_to_name.get(int(ann["image_id"]))
        if name and "bbox" in ann:
            roi_map[name] = ann["bbox"]
            roi_map[Path(name).stem] = ann["bbox"]
    return roi_map


def canonical_bottle_type(value: str) -> int:
    text = str(value).lower()
    if "vichy" in text:
        return 0
    if "euro" in text:
        return 1
    if "nrw" in text:
        return 2
    return -1


def load_btype_map(path: Path) -> dict[str, int]:
    if not path.exists():
        return {}
    df = pd.read_csv(path)
    if "split" in df.columns:
        df = df[df["split"].eq("test")].copy()
    df["btype_id"] = df["bottle_type"].map(canonical_bottle_type)
    out = {}
    for row in df.itertuples(index=False):
        image_id = str(row.image_id)
        out[image_id] = int(row.btype_id)
        out[Path(image_id).stem] = int(row.btype_id)
    return out


def crop_roi(image: np.ndarray, bbox, padding: int = 8) -> np.ndarray:
    if bbox is None:
        return image
    x, y, w, h = bbox
    ih, iw = image.shape[:2]
    x1 = max(0, int(x) - padding)
    y1 = max(0, int(y) - padding)
    x2 = min(iw, int(x + w) + padding)
    y2 = min(ih, int(y + h) + padding)
    if x2 <= x1 or y2 <= y1:
        return image
    return image[y1:y2, x1:x2]


def read_image(image_id: str) -> np.ndarray:
    path = TEST_IMG_DIR / image_id
    if not path.exists():
        stem = Path(image_id).stem
        for ext in [".png", ".jpg", ".jpeg"]:
            candidate = TEST_IMG_DIR / f"{stem}{ext}"
            if candidate.exists():
                path = candidate
                break
    image_bgr = cv2.imread(str(path))
    if image_bgr is None:
        raise FileNotFoundError(f"Could not read test image: {image_id}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def preprocess(image_id: str, roi_map: dict[str, list[float]]) -> np.ndarray:
    image = read_image(image_id)
    cropped = crop_roi(image, roi_map.get(image_id) or roi_map.get(Path(image_id).stem))
    resized = cv2.resize(cropped, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    arr = resized.astype(np.float32) / 255.0
    arr = (arr - NORM_MEAN) / NORM_STD
    arr = np.transpose(arr, (2, 0, 1))
    return arr.astype(np.float32)


def btype_for(image_id: str, btype_map: dict[str, int]) -> int:
    return int(btype_map.get(image_id, btype_map.get(Path(image_id).stem, -1)))


roi_map = load_roi_map(ROI_PATH)
btype_map = load_btype_map(BOTTLE_PATH)
sample = pd.read_csv(SAMPLE_PATH)

print("test rows:", len(sample))
print("ROI entries:", len(roi_map))
print("Bottle type entries:", len(btype_map))
print(sample.head())


test rows: 4418
ROI entries: 8836
Bottle type entries: 8836
                                            image_id  target
0  64e50373-1f39-40e7-b821-655246c7a30e_000000000...       0
1  43b0c8b4-75b6-40b2-b867-ea2779f6ecec_000000000...       0
2  50269c78-2962-42a0-ac21-e83155eaeeb3_000000000...       1
3  a13c8c3d-ace7-4b58-8a25-697e6d3d4bc5_000000000...       0
4  0f06081b-03f9-4179-ab26-cd402f40e43c_000000000...       0


In [5]:
providers = [
    provider
    for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
    if provider in ort.get_available_providers()
]
if not providers:
    providers = ort.get_available_providers()

session = ort.InferenceSession(str(MODEL_PATH), providers=providers)
session_inputs = session.get_inputs()
image_input_name = session_inputs[0].name
btype_input_name = session_inputs[1].name if len(session_inputs) > 1 else None
output_name = session.get_outputs()[0].name

print("using providers:", session.get_providers())
for item in session_inputs:
    print("input:", item.name, item.shape, item.type)
print("output:", output_name, session.get_outputs()[0].shape)


def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def run_batch(batch_ids: list[str]) -> np.ndarray:
    batch = np.stack([preprocess(image_id, roi_map) for image_id in batch_ids], axis=0)
    feed = {image_input_name: batch}
    if btype_input_name is not None:
        feed[btype_input_name] = np.array(
            [btype_for(image_id, btype_map) for image_id in batch_ids],
            dtype=np.int64,
        )
    output = session.run([output_name], feed)[0]
    output = np.asarray(output).reshape(len(batch_ids), -1)[:, 0]
    if OUTPUT_ACTIVATION == "sigmoid":
        output = sigmoid(output)
    elif OUTPUT_ACTIVATION in {"identity", "probability", "none"}:
        output = output.astype(np.float32)
    else:
        raise ValueError(f"Unknown output_activation: {OUTPUT_ACTIVATION}")
    return output.astype(np.float32)


using providers: ['CPUExecutionProvider']
input: image ['batch', 3, 256, 256] tensor(float)
input: btype_id ['batch'] tensor(int64)
output: logits ['batch']


In [6]:
started = time.time()
image_ids = sample["image_id"].astype(str).tolist()
probs = np.zeros(len(image_ids), dtype=np.float32)

for start in range(0, len(image_ids), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(image_ids))
    probs[start:end] = run_batch(image_ids[start:end])
    if start == 0 or end == len(image_ids) or (start // BATCH_SIZE) % 10 == 0:
        elapsed = time.time() - started
        print(f"{end}/{len(image_ids)} images | elapsed {elapsed:.1f}s")

preds = (probs >= THRESHOLD).astype(int)
submission = sample[["image_id"]].copy()
submission["target"] = preds
submission.to_csv("submission.csv", index=False)

elapsed = time.time() - started
print("finished inference")
print("elapsed_seconds:", round(elapsed, 2))
print("mean_probability:", float(probs.mean()))
print("positive_predictions:", int(preds.sum()), "/", len(preds))
print("wrote: submission.csv")
submission.head()


32/4418 images | elapsed 1.1s


352/4418 images | elapsed 12.2s


672/4418 images | elapsed 23.3s


992/4418 images | elapsed 34.5s


1312/4418 images | elapsed 45.9s


...

3552/4418 images | elapsed 129.4s


3872/4418 images | elapsed 140.9s


4192/4418 images | elapsed 153.2s


4418/4418 images | elapsed 161.3s
finished inference
elapsed_seconds: 161.33
mean_probability: 0.5708510279655457
positive_predictions: 2527 / 4418
wrote: submission.csv


,image_id,target
0,64e50373-1f39-40e7-b821-655246c7a30e_000000000...,1
1,43b0c8b4-75b6-40b2-b867-ea2779f6ecec_000000000...,0
2,50269c78-2962-42a0-ac21-e83155eaeeb3_000000000...,0
3,a13c8c3d-ace7-4b58-8a25-697e6d3d4bc5_000000000...,0
4,0f06081b-03f9-4179-ab26-cd402f40e43c_000000000...,1
